In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [2]:
import numpy as np
import pyreadr
from pathlib import Path
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
import geopandas as gpd

# ================================================================
# 0. CONFIG
# ================================================================

BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# 1. LOAD DATA (NON-ISOLATED ONLY)
# ================================================================

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

y_full = snow.iloc[:, 2:].to_numpy()
coords_full = snow.iloc[:, :2].to_numpy()

y = np.delete(y_full, no_nbs, axis=0)
coords = np.delete(coords_full, no_nbs, axis=0)

S, T = y.shape
print(f"S = {S}, T = {T}")

# ================================================================
# 2. BUILD SPATIAL ADJACENCY (MATCH MODEL)
# ================================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
rotated = (xy @ R.T) / 1e6

Distances = squareform(pdist(rotated))
W = (Distances <= 0.22).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

# ================================================================
# 3. OBSERVED STATISTIC (GLOBAL)
# ================================================================

T_obs = 0.0

# t = 2,...,T-1  <-> Python t = 1,...,T-2
for t in range(1, T-1):

    # space neighbors
    space_sum = (W @ y[:, t]).sum()

    # time neighbors (same cell)
    time_sum = y[:, t-1].sum() + y[:, t+1].sum()

    T_obs += space_sum + time_sum

print("Observed T =", T_obs)

# ================================================================
# 4. LOAD POSTERIOR SAMPLES
# ================================================================

bym01 = np.load(BASE_DIR / "bym01_noIso_final.npz")["all_theta"]
bym10 = np.load(BASE_DIR / "bym10_noIso_final.npz")["all_theta"]

ind01_full = np.load(BASE_DIR / "ind01.npz")["all_theta"]
ind10_full = np.load(BASE_DIR / "ind10.npz")["all_theta"]

M = bym01.shape[1]
print(f"M = {M}")

# extract non-isolated IID parameters
S_full = y_full.shape[0]
K_iid = 4

ind01 = np.zeros((K_iid*S, M))
ind10 = np.zeros((K_iid*S, M))

for k in range(K_iid):
    ind01[k*S:(k+1)*S, :] = ind01_full[k*S_full:k*S_full+S, :]
    ind10[k*S:(k+1)*S, :] = ind10_full[k*S_full:k*S_full+S, :]

# ================================================================
# 5. BUILD COVARIATES (SCALED TIME — MATCH MCMC)
# ================================================================

t_raw = np.arange(1, T+1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std()

cov_bym = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
    t_trend, t_trend
])

cov_iid = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period),
    t_trend
])

# ================================================================
# 6. PPC — BYM (FORWARD SIMULATION)
# ================================================================

T_rep_bym = np.zeros(M)

print("\nRunning PPC Option 5 (BYM)...")

for m in tqdm(range(M), desc="BYM space–time PPC"):

    y_rep = np.zeros((S, T), dtype=int)
    y_rep[:, 0] = y[:, 0]

    # simulate full trajectory
    for t in range(1, T):

        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(8):
            eta01 += cov_bym[t, k] * bym01[k*S:(k+1)*S, m]
            eta10 += cov_bym[t, k] * bym10[k*S:(k+1)*S, m]

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))

        prob = np.where(
            y_rep[:, t-1] == 0,
            p01,
            1 - p10
        )

        y_rep[:, t] = np.random.binomial(1, prob)

    # compute GLOBAL statistic
    Tm = 0.0
    for t in range(1, T-1):
        space_sum = (W @ y_rep[:, t]).sum()
        time_sum = y_rep[:, t-1].sum() + y_rep[:, t+1].sum()
        Tm += space_sum + time_sum

    T_rep_bym[m] = Tm

# ================================================================
# 7. PPC — IID (FORWARD SIMULATION)
# ================================================================

T_rep_iid = np.zeros(M)

print("\nRunning PPC Option 5 (IID)...")

for m in tqdm(range(M), desc="IID space–time PPC"):

    y_rep = np.zeros((S, T), dtype=int)
    y_rep[:, 0] = y[:, 0]

    for t in range(1, T):

        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(4):
            eta01 += cov_iid[t, k] * ind01[k*S:(k+1)*S, m]
            eta10 += cov_iid[t, k] * ind10[k*S:(k+1)*S, m]

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))

        prob = np.where(
            y_rep[:, t-1] == 0,
            p01,
            1 - p10
        )

        y_rep[:, t] = np.random.binomial(1, prob)

    Tm = 0.0
    for t in range(1, T-1):
        space_sum = (W @ y_rep[:, t]).sum()
        time_sum = y_rep[:, t-1].sum() + y_rep[:, t+1].sum()
        Tm += space_sum + time_sum

    T_rep_iid[m] = Tm

# ================================================================
# 8. POSTERIOR P-VALUES (SCALAR)
# ================================================================

p_bym = np.mean(T_rep_bym >= T_obs)
p_iid = np.mean(T_rep_iid >= T_obs)

print("\n===== PPC Option 5 (GLOBAL SPACE–TIME) =====\n")
print(f"BYM p-value: {p_bym:.4f}")
print(f"IID p-value: {p_iid:.4f}")

# ================================================================
# 9. SAVE
# ================================================================

np.savez(
    BASE_DIR / "ppc_space_time_neighbor_global.npz",
    T_obs=T_obs,
    T_rep_bym=T_rep_bym,
    T_rep_iid=T_rep_iid,
    p_bym=p_bym,
    p_iid=p_iid
)

print("\nSaved to ppc_space_time_neighbor_global.npz")


S = 1601, T = 2704
Observed T = 8479668.0
M = 1000

Running PPC Option 5 (BYM)...


BYM space–time PPC: 100%|██████████| 1000/1000 [10:23<00:00,  1.60it/s]



Running PPC Option 5 (IID)...


IID space–time PPC: 100%|██████████| 1000/1000 [08:11<00:00,  2.03it/s]


===== PPC Option 5 (GLOBAL SPACE–TIME) =====

BYM p-value: 0.0000
IID p-value: 1.0000

Saved to ppc_space_time_neighbor_global.npz
